# CADLabs vs. Equivocation Comparison

This notebook compares two complementary modeling approaches used in our project:

- the CADLabs / Hoban-Borgers economic model, which estimates validator revenue, profit, and slashing drag under behavior scenarios
- a local equivocation / accountable-safety model, which studies when collusive behavior can threaten finality and how much slashable stake is required to restore safety

The comparison is intentionally **overlap-only**. We do not force unsupported one-to-one metrics. Instead, we show how the two methods answer different parts of the same collusion question.

In [1]:
import setup_path
import setup_templates

from dataclasses import dataclass
from pathlib import Path
import copy
import logging
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
from IPython.display import Markdown, display

from experiments.run import run
import experiments.templates.time_domain_analysis as time_domain_analysis

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
preferred_style = "seaborn-v0_8-whitegrid"
fallback_style = "ggplot"
plt.style.use(preferred_style if preferred_style in plt.style.available else fallback_style)
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
logging.getLogger().setLevel(logging.CRITICAL)

COLORS = {
    "honest": "#2f7f6f",
    "colluding": "#c55353",
    "offline": "#d49c3d",
    "pool": "#4d79a7",
}

CADLABS_ROOT = Path(setup_templates.__file__).resolve().parents[1]
REPO_ROOT = CADLABS_ROOT.parent
RESULTS_ROOT = REPO_ROOT / "shared" / "comparison"
TABLES_DIR = RESULTS_ROOT / "tables"
CHARTS_DIR = RESULTS_ROOT / "charts"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)


def save_table(frame: pd.DataFrame, filename: str) -> Path:
    target = TABLES_DIR / filename
    frame.to_csv(target, index=False)
    return target


def save_figure(fig: plt.Figure, filename: str) -> Path:
    target = CHARTS_DIR / filename
    fig.savefig(target, bbox_inches="tight")
    return target


display(Markdown(f"Comparison results will be exported to `{RESULTS_ROOT}`."))

Comparison results will be exported to `C:\Users\jrh25\Desktop\staking-economics\shared\comparison`.

## Methodology Note

The CADLabs section uses the same four behavior profiles from our earlier experiment notebook: `honest`, `colluding`, `offline`, and `pool`. The equivocation section uses a normalized 1,000-validator population with a 34% attacker share, which is just above the one-third accountable-safety threshold implied by a 2/3 finality rule. That normalized setup keeps the slashing and finality plots readable while preserving the protocol logic.

To keep the notebook runnable inside the existing CADLabs environment, the equivocation logic is reproduced locally below from our repo-native implementation.

In [2]:
@dataclass(frozen=True)
class EquivocationAttackConfig:
    total_validators: int
    attacker_fraction: float = 0.34
    honest_partition_fraction: float = 0.50
    finality_threshold: float = 2 / 3
    epochs: int = 8
    slash_detection_delay_epochs: int = 1
    slash_detection_fraction_per_epoch: float = 1.0
    slash_fraction_of_balance: float = 1.0
    validator_balance_eth: float = 32.0
    attack_sweep_max_fraction: float = 0.50
    attack_sweep_points: int = 26


def accountable_safety_bound_fraction(finality_threshold: float) -> float:
    return max(0.0, min(1.0, (2.0 * finality_threshold) - 1.0))


def branch_vote_shares(*, honest_validators: int, attacker_validators: int, honest_partition_fraction: float):
    total_validators = honest_validators + attacker_validators
    if total_validators <= 0:
        return 0.0, 0.0

    honest_branch_a = honest_validators * honest_partition_fraction
    honest_branch_b = honest_validators - honest_branch_a
    branch_a_share = (honest_branch_a + attacker_validators) / total_validators
    branch_b_share = (honest_branch_b + attacker_validators) / total_validators
    return branch_a_share, branch_b_share


def minimum_slashed_to_restore_safety(*, attacker_validators: int, total_validators: int, bound_fraction: float) -> int:
    if attacker_validators <= 0 or total_validators <= 0 or bound_fraction <= 0:
        return 0
    if bound_fraction >= 1.0:
        return attacker_validators

    numerator = attacker_validators - (bound_fraction * total_validators)
    if numerator < 0:
        return 0

    denominator = 1.0 - bound_fraction
    required = math.floor(numerator / denominator) + 1
    return max(0, min(attacker_validators, required))


def build_equivocation_snapshot(config: EquivocationAttackConfig):
    bound_fraction = accountable_safety_bound_fraction(config.finality_threshold)
    honest_validators = max(
        config.total_validators - round(config.total_validators * config.attacker_fraction),
        0,
    )
    attacker_validators = max(config.total_validators - honest_validators, 0)

    epoch_rows = []
    cumulative_slashed = 0
    cumulative_burned_stake_eth = 0.0

    for epoch in range(max(config.epochs, 1) + 1):
        active_total = honest_validators + attacker_validators
        attacker_share = attacker_validators / active_total if active_total > 0 else 0.0
        branch_a_share, branch_b_share = branch_vote_shares(
            honest_validators=honest_validators,
            attacker_validators=attacker_validators,
            honest_partition_fraction=config.honest_partition_fraction,
        )
        conflict_possible = (
            attacker_validators > 0
            and branch_a_share >= config.finality_threshold
            and branch_b_share >= config.finality_threshold
        )
        minimum_slashed = minimum_slashed_to_restore_safety(
            attacker_validators=attacker_validators,
            total_validators=active_total,
            bound_fraction=bound_fraction,
        )

        slashed_this_epoch = 0
        if epoch >= config.slash_detection_delay_epochs and attacker_validators > 0:
            slashed_this_epoch = math.ceil(
                attacker_validators * config.slash_detection_fraction_per_epoch
            )
            slashed_this_epoch = max(0, min(attacker_validators, slashed_this_epoch))

        post_attackers = attacker_validators - slashed_this_epoch
        post_total = honest_validators + post_attackers
        post_attacker_share = post_attackers / post_total if post_total > 0 else 0.0

        cumulative_slashed += slashed_this_epoch
        cumulative_burned_stake_eth += (
            slashed_this_epoch * config.validator_balance_eth * config.slash_fraction_of_balance
        )

        epoch_rows.append(
            {
                "epoch": epoch,
                "attacker_share_pct_before_slash": attacker_share * 100.0,
                "branch_a_vote_share_pct_before_slash": branch_a_share * 100.0,
                "branch_b_vote_share_pct_before_slash": branch_b_share * 100.0,
                "conflicting_finalization_possible": conflict_possible,
                "minimum_slashed_to_restore_safety": minimum_slashed,
                "slashed_this_epoch": slashed_this_epoch,
                "cumulative_slashed_validators": cumulative_slashed,
                "cumulative_burned_stake_eth": cumulative_burned_stake_eth,
                "attacker_share_pct_after_slash": post_attacker_share * 100.0,
                "safety_restored_after_slash": post_attacker_share < (bound_fraction - 1e-12),
            }
        )

        attacker_validators = post_attackers

    sweep_rows = []
    for index in range(max(config.attack_sweep_points, 2)):
        fraction = config.attack_sweep_max_fraction * index / (max(config.attack_sweep_points, 2) - 1)
        attacker_validators = round(config.total_validators * fraction)
        honest_validators = max(config.total_validators - attacker_validators, 0)
        branch_a_share, branch_b_share = branch_vote_shares(
            honest_validators=honest_validators,
            attacker_validators=attacker_validators,
            honest_partition_fraction=config.honest_partition_fraction,
        )
        conflict_possible = (
            attacker_validators > 0
            and branch_a_share >= config.finality_threshold
            and branch_b_share >= config.finality_threshold
        )
        minimum_slashed = minimum_slashed_to_restore_safety(
            attacker_validators=attacker_validators,
            total_validators=config.total_validators,
            bound_fraction=bound_fraction,
        )
        post_attackers = attacker_validators - minimum_slashed
        post_total = config.total_validators - minimum_slashed
        post_attacker_share = post_attackers / post_total if post_total > 0 else 0.0
        sweep_rows.append(
            {
                "attacker_share_pct": fraction * 100.0,
                "conflicting_finalization_possible": conflict_possible,
                "minimum_slashed_to_restore_safety": minimum_slashed,
                "minimum_slashed_to_restore_safety_pct": minimum_slashed / config.total_validators * 100.0,
                "post_response_attacker_share_pct": post_attacker_share * 100.0,
            }
        )

    epoch_frame = pd.DataFrame(epoch_rows)
    sweep_frame = pd.DataFrame(sweep_rows)
    first_row = epoch_frame.iloc[0]
    restored_epochs = epoch_frame.loc[epoch_frame["safety_restored_after_slash"], "epoch"]
    first_restored_epoch = int(restored_epochs.iloc[0]) if not restored_epochs.empty else None
    final_row = epoch_frame.iloc[-1]

    summary = {
        "initial_attacker_share_pct": float(first_row["attacker_share_pct_before_slash"]),
        "initial_conflicting_finalization_possible": bool(first_row["conflicting_finalization_possible"]),
        "initial_minimum_slashed_to_restore_safety": int(first_row["minimum_slashed_to_restore_safety"]),
        "final_cumulative_burned_stake_eth": float(final_row["cumulative_burned_stake_eth"]),
        "first_restored_epoch": first_restored_epoch,
    }

    return summary, epoch_frame, sweep_frame

In [3]:
SCENARIO_LABELS = {0: "honest", 1: "colluding", 2: "offline", 3: "pool"}


def latest_substep_frame(frame: pd.DataFrame) -> pd.DataFrame:
    latest_substep = int(frame["substep"].max())
    latest = frame.loc[frame["substep"] == latest_substep].copy()
    return latest.sort_values(["subset", "run", "timestep"]).reset_index(drop=True)


def build_cadlabs_frames():
    simulation = copy.deepcopy(time_domain_analysis.experiment.simulations[0])
    simulation.runs = 1
    simulation.model.params.update(
        {
            "validator_uptime_process": [
                lambda _run, _timestep: 0.98,
                lambda _run, _timestep: 0.80,
                lambda _run, _timestep: 2 / 3,
                lambda _run, _timestep: 0.95,
            ],
            "slashing_events_per_1000_epochs": [0, 5, 0, 1],
            "validator_third_party_costs_per_epoch": [
                [0.0] * 7,
                [0.0] * 7,
                [0.0] * 7,
                [0.10] * 7,
            ],
        }
    )

    raw_frame, exceptions = run(simulation)
    actual_exceptions = [item for item in exceptions if item.get("exception") is not None]
    if actual_exceptions:
        raise RuntimeError(actual_exceptions[0])

    latest = latest_substep_frame(raw_frame)
    latest["scenario"] = latest["subset"].map(SCENARIO_LABELS)

    cumulative_slashing = (
        latest.groupby("subset", as_index=False)["amount_slashed_eth"]
        .sum()
        .rename(columns={"amount_slashed_eth": "cumulative_slashed_eth"})
    )

    summary = (
        latest.sort_values(["subset", "timestep"])
        .groupby(["subset", "scenario"], as_index=False)
        .tail(1)
        .merge(cumulative_slashing, on="subset", how="left")
        .rename(
            columns={
                "number_of_active_validators": "final_active_validators",
                "total_revenue_yields_pct": "final_revenue_yield_pct",
                "total_profit_yields_pct": "final_profit_yield_pct",
                "revenue_profit_yield_spread_pct": "yield_spread_pct",
            }
        )
        [[
            "subset",
            "scenario",
            "final_active_validators",
            "final_revenue_yield_pct",
            "final_profit_yield_pct",
            "yield_spread_pct",
            "cumulative_slashed_eth",
        ]]
        .sort_values("subset")
        .reset_index(drop=True)
    )

    return raw_frame, latest, summary


cadlabs_raw, cadlabs_latest, cadlabs_summary = build_cadlabs_frames()
save_table(cadlabs_latest, "cadlabs_latest_substep_timeseries.csv")
save_table(cadlabs_summary, "cadlabs_summary.csv")
cadlabs_summary

,subset,scenario,final_active_validators,final_revenue_yield_pct,final_profit_yield_pct,yield_spread_pct,cumulative_slashed_eth
0,0,honest,885250,3.23,3.13,0.11,0.00
1,1,colluding,885250,1.81,1.70,0.11,607.50
2,2,offline,885250,0.86,0.76,0.11,0.00
3,3,pool,885250,2.98,2.58,0.40,121.50


In [4]:
display(Markdown("## CADLabs Results"))
cadlabs_summary_display = cadlabs_summary.rename(
        columns={
            "scenario": "Scenario",
            "final_active_validators": "Final active validators",
            "final_revenue_yield_pct": "Revenue yield (%)",
            "final_profit_yield_pct": "Profit yield (%)",
            "yield_spread_pct": "Revenue-profit spread (ppt)",
            "cumulative_slashed_eth": "Cumulative slashed ETH",
        }
    )[[
        "Scenario",
        "Final active validators",
        "Revenue yield (%)",
        "Profit yield (%)",
        "Revenue-profit spread (ppt)",
        "Cumulative slashed ETH",
    ]]
display(cadlabs_summary_display)
save_table(cadlabs_summary_display, "cadlabs_summary_display.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

scenario_positions = range(len(cadlabs_summary))
bar_width = 0.36
revenue_values = cadlabs_summary["final_revenue_yield_pct"].tolist()
profit_values = cadlabs_summary["final_profit_yield_pct"].tolist()

axes[0].bar(
    [position - bar_width / 2 for position in scenario_positions],
    revenue_values,
    width=bar_width,
    color="#7fb8ad",
    label="Revenue yield",
)
axes[0].bar(
    [position + bar_width / 2 for position in scenario_positions],
    profit_values,
    width=bar_width,
    color="#356d69",
    label="Profit yield",
)
axes[0].set_xticks(list(scenario_positions))
axes[0].set_xticklabels(cadlabs_summary["scenario"].str.title())
axes[0].set_ylabel("Annualized yield (%)")
axes[0].set_title("Final Yields by Behavior")
axes[0].legend(frameon=False)

for scenario, frame in cadlabs_latest.groupby("scenario"):
    axes[1].plot(
        frame["timestep"],
        frame["total_profit_yields_pct"],
        label=scenario.title(),
        color=COLORS[scenario],
        linewidth=2.2,
    )

axes[1].axhline(0.0, color="#444444", linewidth=1.0, linestyle="--")
axes[1].set_xlabel("Timestep")
axes[1].set_ylabel("Annualized profit yield (%)")
axes[1].set_title("Profit Yield Over Time")
axes[1].legend(frameon=False)

plt.suptitle("CADLabs Economic Model", fontsize=14, y=1.02)
plt.tight_layout()
save_figure(fig, "cadlabs_economic_model.png")
plt.show()

## CADLabs Results

,Scenario,Final active validators,Revenue yield (%),Profit yield (%),Revenue-profit spread (ppt),Cumulative slashed ETH
0,honest,885250,3.23,3.13,0.11,0.00
1,colluding,885250,1.81,1.70,0.11,607.50
2,offline,885250,0.86,0.76,0.11,0.00
3,pool,885250,2.98,2.58,0.40,121.50


C:\Users\jrh25\Desktop\staking-economics\shared\output\ipykernel_63220\451522208.py:67: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Equivocation / Accountable-Safety Results

The next section switches from long-run validator profitability to the consensus safety side of the collusion problem. This model does not estimate APY. Instead, it tracks whether an equivocating attacker can support conflicting supermajorities and how much slashable stake must be removed to restore safety.

In [5]:
equivocation_config = EquivocationAttackConfig(
    total_validators=1_000,
    attacker_fraction=0.34,
)
equivocation_summary_dict, equivocation_epoch, equivocation_sweep = build_equivocation_snapshot(equivocation_config)

first_conflict_row = equivocation_sweep.loc[
    equivocation_sweep["conflicting_finalization_possible"]
].head(1)
first_conflict_share = (
    float(first_conflict_row["attacker_share_pct"].iloc[0]) if not first_conflict_row.empty else float("nan")
)

equivocation_summary = pd.DataFrame(
    [
        {
            "Metric": "Initial attacker share (%)",
            "Value": equivocation_summary_dict["initial_attacker_share_pct"],
        },
        {
            "Metric": "Conflicting finalization feasible at start",
            "Value": str(equivocation_summary_dict["initial_conflicting_finalization_possible"]),
        },
        {
            "Metric": "Minimum slashed to restore safety (validators)",
            "Value": equivocation_summary_dict["initial_minimum_slashed_to_restore_safety"],
        },
        {
            "Metric": "First attacker share where conflict becomes feasible (%)",
            "Value": first_conflict_share,
        },
        {
            "Metric": "Cumulative burned stake after response (ETH)",
            "Value": equivocation_summary_dict["final_cumulative_burned_stake_eth"],
        },
        {
            "Metric": "Epoch when safety is first restored",
            "Value": equivocation_summary_dict["first_restored_epoch"],
        },
    ]
)

save_table(equivocation_summary, "equivocation_summary.csv")
save_table(equivocation_epoch, "equivocation_epoch.csv")
save_table(equivocation_sweep, "equivocation_sweep.csv")

equivocation_summary

,Metric,Value
0,Initial attacker share (%),34.00
1,Conflicting finalization feasible at start,True
2,Minimum slashed to restore safety (validators),11
3,First attacker share where conflict becomes feasible (%),34.00
4,Cumulative burned stake after response (ETH),"10,880.00"
5,Epoch when safety is first restored,1


In [6]:
display(equivocation_summary)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.plot(
    equivocation_sweep["attacker_share_pct"],
    equivocation_sweep["minimum_slashed_to_restore_safety_pct"],
    color="#a84747",
    linewidth=2.4,
)
ax.fill_between(
    equivocation_sweep["attacker_share_pct"],
    0,
    equivocation_sweep["minimum_slashed_to_restore_safety_pct"],
    where=equivocation_sweep["conflicting_finalization_possible"],
    color="#e6a4a4",
    alpha=0.45,
    label="Conflict-feasible region",
)
ax.axvline(100 / 3, color="#555555", linestyle="--", linewidth=1.2, label="One-third threshold")
ax.set_xlabel("Attacker share of validators (%)")
ax.set_ylabel("Minimum slashed to restore safety (%)")
ax.set_title("Equivocation Sweep: Conflict Feasibility and Required Slashing")
ax.yaxis.set_major_formatter(PercentFormatter())
ax.legend(frameon=False)
plt.tight_layout()
save_figure(fig, "equivocation_sweep.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(
    equivocation_epoch["epoch"],
    equivocation_epoch["attacker_share_pct_before_slash"],
    color="#c55353",
    linewidth=2.2,
    label="Before slashing",
)
axes[0].plot(
    equivocation_epoch["epoch"],
    equivocation_epoch["attacker_share_pct_after_slash"],
    color="#356d69",
    linewidth=2.2,
    label="After slashing",
)
axes[0].axhline(100 / 3, color="#555555", linestyle="--", linewidth=1.2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Attacker share (%)")
axes[0].set_title("Attacker Share Through the Slashing Response")
axes[0].yaxis.set_major_formatter(PercentFormatter())
axes[0].legend(frameon=False)

axes[1].bar(
    equivocation_epoch["epoch"],
    equivocation_epoch["cumulative_burned_stake_eth"],
    color="#c98c46",
    width=0.7,
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cumulative burned stake (ETH)")
axes[1].set_title("Burned Stake Accumulation During Safety Restoration")

plt.suptitle("Equivocation / Accountable-Safety Model", fontsize=14, y=1.02)
plt.tight_layout()
save_figure(fig, "equivocation_accountable_safety_model.png")
plt.show()

,Metric,Value
0,Initial attacker share (%),34.00
1,Conflicting finalization feasible at start,True
2,Minimum slashed to restore safety (validators),11
3,First attacker share where conflict becomes feasible (%),34.00
4,Cumulative burned stake after response (ETH),"10,880.00"
5,Epoch when safety is first restored,1


C:\Users\jrh25\Desktop\staking-economics\shared\output\ipykernel_63220\2405045274.py:27: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


C:\Users\jrh25\Desktop\staking-economics\shared\output\ipykernel_63220\2405045274.py:64: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Cross-Method Comparison

The table below compares the two methods on the common collusion theme while keeping the interpretation honest about what each model does and does not simulate.

In [7]:
honest_row = cadlabs_summary.loc[cadlabs_summary["scenario"] == "honest"].iloc[0]
colluding_row = cadlabs_summary.loc[cadlabs_summary["scenario"] == "colluding"].iloc[0]

profit_penalty = honest_row["final_profit_yield_pct"] - colluding_row["final_profit_yield_pct"]
revenue_penalty = honest_row["final_revenue_yield_pct"] - colluding_row["final_revenue_yield_pct"]
initial_min_slashed = equivocation_summary_dict["initial_minimum_slashed_to_restore_safety"]
initial_min_slashed_pct = 100.0 * initial_min_slashed / equivocation_config.total_validators

comparison_frame = pd.DataFrame(
    [
        {
            "Question": "What is the cost of collusive behavior?",
            "CADLabs answer": (
                f"The colluding profile ends at {colluding_row['final_profit_yield_pct']:.2f}% profit yield versus "
                f"{honest_row['final_profit_yield_pct']:.2f}% for honest validators, a {profit_penalty:.2f} ppt profit penalty."
            ),
            "Equivocation answer": "This model does not annualize yield; it treats collusion as a safety problem and measures slashable stake instead of validator APY.",
        },
        {
            "Question": "When does collusion threaten finality?",
            "CADLabs answer": (
                "The economic model proxies collusion through lower uptime and expected slashing frequency, "
                "but it does not simulate conflicting finalization directly."
            ),
            "Equivocation answer": (
                f"Conflicting finalization first becomes feasible at about {first_conflict_share:.1f}% attacker share "
                "under a two-thirds supermajority rule."
            ),
        },
        {
            "Question": "How much slashing is needed to restore safety?",
            "CADLabs answer": (
                f"The colluding scenario accumulates {colluding_row['cumulative_slashed_eth']:.2f} ETH of modeled slashing loss "
                "over the simulation horizon."
            ),
            "Equivocation answer": (
                f"At the default 34% attacker share, at least {initial_min_slashed} validators "
                f"({initial_min_slashed_pct:.2f}% of the network) must be slashed to push the attacker below the accountable-safety bound."
            ),
        },
    ]
)

save_table(comparison_frame, "cross_method_comparison.csv")

comparison_frame

,Question,CADLabs answer,Equivocation answer
0,What is the cost of collusive behavior?,"The colluding profile ends at 1.70% profit yield versus 3.13% for honest validators, a 1.42 ppt profit penalty.",This model does not annualize yield; it treats collusion as a safety problem and measures slashable stake instead of...
1,When does collusion threaten finality?,"The economic model proxies collusion through lower uptime and expected slashing frequency, but it does not simulate ...",Conflicting finalization first becomes feasible at about 34.0% attacker share under a two-thirds supermajority rule.
2,How much slashing is needed to restore safety?,The colluding scenario accumulates 607.50 ETH of modeled slashing loss over the simulation horizon.,"At the default 34% attacker share, at least 11 validators (1.10% of the network) must be slashed to push the attacke..."


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))

axes[0].bar(
    ["Honest revenue", "Colluding revenue", "Honest profit", "Colluding profit"],
    [
        honest_row["final_revenue_yield_pct"],
        colluding_row["final_revenue_yield_pct"],
        honest_row["final_profit_yield_pct"],
        colluding_row["final_profit_yield_pct"],
    ],
    color=["#8bc1b7", "#e39696", "#2f7f6f", "#c55353"],
)
axes[0].set_ylabel("Annualized yield (%)")
axes[0].set_title(
    f"CADLabs: Honest vs. Colluding Yield Gap\nRevenue gap {revenue_penalty:.2f} ppt, profit gap {profit_penalty:.2f} ppt"
)
axes[0].tick_params(axis="x", rotation=18)

axes[1].plot(
    equivocation_sweep["attacker_share_pct"],
    equivocation_sweep["minimum_slashed_to_restore_safety_pct"],
    color="#a84747",
    linewidth=2.4,
)
axes[1].axvline(100 / 3, color="#555555", linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Attacker share (%)")
axes[1].set_ylabel("Minimum slashed to restore safety (%)")
axes[1].set_title("Equivocation: Safety Restoration Cost")
axes[1].yaxis.set_major_formatter(PercentFormatter())

plt.suptitle("Collusion Through Two Lenses", fontsize=14, y=1.02)
plt.tight_layout()
save_figure(fig, "collusion_through_two_lenses.png")
plt.show()

C:\Users\jrh25\Desktop\staking-economics\shared\output\ipykernel_63220\928651234.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Interpretation

These outputs should be read as **complementary rather than numerically identical**. The CADLabs model explains how collusion, downtime, and service fees compress validator profitability over time. The equivocation model explains when collusion crosses from an economic underperformance problem into a protocol safety problem, and how much slashable stake must be removed to restore accountable safety. Together, they show both the **economic penalty** and the **consensus-security consequence** of degraded validator behavior.